In [2]:
# Cell 1: Install packages (run first)
!pip install -q --no-cache-dir peft bitsandbytes accelerate transformers datasets

import warnings
warnings.filterwarnings("ignore")

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

print("GPU being used:", torch.cuda.get_device_name(0))

GPU being used: Tesla T4


In [3]:
# Cell 2: Load the open model
model_name = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Base model loaded successfully!")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Base model loaded successfully!


In [4]:
# Cell 3: Add LoRA adapter
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

print("LoRA adapter attached.")

LoRA adapter attached.


In [8]:
# Cell 4: Fixed Quick Test for Qwen2.5 (safe extraction)
def run_quick_test():
    prompt = """Here are some examples of a pattern:
Input: 2, Output: 4
Input: 3, Output: 6
What should be the output for Input: 5?

Think carefully step by step and put your final answer inside \\boxed{}."""

    messages = [
        {"role": "system", "content": "You are an expert at finding hidden patterns and rules. Always think step by step before answering. Put the final answer in \\boxed{}."},
        {"role": "user", "content": prompt}
    ]

    # Safe way for Qwen2.5
    input_text = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )

    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    outputs = model.generate(
        inputs.input_ids,
        max_new_tokens=512,
        temperature=0.0,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print("\n=== Model Response ===")
    print(response)
    print("======================")

run_quick_test()

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



=== Model Response ===
system
You are an expert at finding hidden patterns and rules. Always think step by step before answering. Put the final answer in \boxed{}.
user
Here are some examples of a pattern:
Input: 2, Output: 4
Input: 3, Output: 6
What should be the output for Input: 5?

Think carefully step by step and put your final answer inside \boxed{}.
assistant
Let's analyze the given examples step by step to identify the pattern:

1. **Identify the relationship between input and output:**
   - For Input: 2, Output: 4
   - For Input: 3, Output: 6

2. **Determine the operation:**
   - Notice that \(2 \times 2 = 4\)
   - Notice that \(3 \times 2 = 6\)

3. **Formulate the rule:**
   - The output is always twice the input.

4. **Apply the rule to the new input:**
   - For Input: 5, the output would be \(5 \times 2 = 10\).

Therefore, the output for Input: 5 is \(\boxed{10}\).


In [11]:
# Corrected Block 2 - Load the training data
import pandas as pd

train_df = pd.read_csv("/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv")

print(f"Training data shape: {train_df.shape}")
print(f"Columns: {train_df.columns.tolist()}")
print("\nFirst 3 examples:")
print(train_df.head(3))

Training data shape: (9500, 3)
Columns: ['id', 'prompt', 'answer']

First 3 examples:
         id                                             prompt  \
0  00066667  In Alice's Wonderland, a secret bit manipulati...   
1  000b53cf  In Alice's Wonderland, a secret bit manipulati...   
2  00189f6a  In Alice's Wonderland, secret encryption rules...   

              answer  
0           10010111  
1           01000011  
2  cat imagines book  


In [14]:
# Block 3: Create the formatting function (this is the key step for training)

def format_example(row):
    # System prompt that forces step-by-step thinking and boxed answer
    system_prompt = """You are a world-class logical reasoner. Always use step-by-step thinking inside <think>...</think> tags before giving the final answer. Put the final answer inside \\boxed{}."""

    # The puzzle is in the 'prompt' column
    user_prompt = row['prompt']

    # The ground truth answer is in the 'answer' column
    # We wrap it in <think> + boxed format for training
    assistant_response = f"""<think>
I will carefully analyze the hidden transformation rule from the examples provided in the puzzle.
I will test different possible rules step by step until I find the one that fits all examples.
</think>

The hidden rule is applied correctly, so the final answer is \\boxed{{{row['answer']}}}"""

    # Combine into the full chat format that Qwen expects
    full_text = f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{user_prompt}<|im_end|>\n<|im_start|>assistant\n{assistant_response}<|im_end|>"

    return full_text

# Test the formatting on the first example
example_formatted = format_example(train_df.iloc[0])
print("=== Example formatted training text (first 800 characters) ===\n")
print(example_formatted[:800] + "...")
print("\n=== Full length:", len(example_formatted), "characters ===")

=== Example formatted training text (first 800 characters) ===

<|im_start|>system
You are a world-class logical reasoner. Always use step-by-step thinking inside <think>...</think> tags before giving the final answer. Put the final answer inside \boxed{}.<|im_end|>
<|im_start|>user
In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.

Here are some examples of input -> output:
01010001 -> 11011101
00001001 -> 01101101
00010101 -> 01010101
11111111 -> 10000001
10011101 -> 01000101
00111011 -> 00001001
10111101 -> 00000101
00100110 -> 10110011

Now, determine the output for: 00110100<|im_end|>
<|im_start|>assistant
<think>
I will carefully analyze the hidden transformation rule from the exampl...

=== Full length: 1019 characters ===


In [22]:
# Fixed Block 4: Training (safe for T4 x 2)
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# Prepare dataset
if 'text' not in train_df.columns:
    train_df['text'] = train_df.apply(format_example, axis=1)

dataset = Dataset.from_pandas(train_df[['text']])

print(f"Dataset created with {len(dataset)} examples")

training_args = TrainingArguments(
    output_dir="qwen_lora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,   # smaller to fit memory safely
    learning_rate=2e-4,
    max_steps=100,
    warmup_steps=10,
    fp16=False,                     # <-- turned off to avoid the error
    logging_steps=10,
    save_strategy="no",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
)

print("Starting training...")
trainer.train()

print("Training finished! LoRA adapter is now updated.")

Dataset created with 9500 examples


Adding EOS to train dataset:   0%|          | 0/9500 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/9500 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/9500 [00:00<?, ? examples/s]

Starting training...


Step,Training Loss
10,2.180160
20,0.953452
30,0.686588
40,0.417327
50,0.542564
60,0.922929
70,0.726288
80,0.548272
90,0.489724
100,0.731783


Training finished! LoRA adapter is now updated.


In [23]:
# Block 5: Save LoRA adapter and create submission.zip

# Save the adapter (this is what Kaggle merges with the base model)
model.save_pretrained("my_lora_adapter")
tokenizer.save_pretrained("my_lora_adapter")

# Create the exact submission file required by the competition
!zip -r submission.zip my_lora_adapter

print("✅ submission.zip created successfully!")
print("File size:")
!ls -lh submission.zip

  adding: my_lora_adapter/ (stored 0%)
  adding: my_lora_adapter/README.md (deflated 65%)
  adding: my_lora_adapter/adapter_model.safetensors (deflated 22%)
  adding: my_lora_adapter/adapter_config.json (deflated 58%)
  adding: my_lora_adapter/tokenizer_config.json (deflated 59%)
  adding: my_lora_adapter/tokenizer.json (deflated 81%)
  adding: my_lora_adapter/chat_template.jinja (deflated 71%)
✅ submission.zip created successfully!
File size:
-rw-r--r-- 1 root root 123M Mar 23 21:50 submission.zip
